# 06 — Chain of Thought (CoT) Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Standard vs CoT: Math Problem

In [3]:
problem = "A store sells apples for $0.50 each. If I buy 12 apples and pay with a $10 bill, how much change do I get?"

standard = chat([{"role":"user","content":problem}], temperature=0, max_tokens=30)
print("Standard:", standard.strip())

cot = chat([{"role":"user","content": problem + "\n\nThink step by step."}],
           temperature=0, max_tokens=200)
print("\nCoT:")
print(cot.strip())


Standard: To find out how much change you get, first, calculate the total cost of the apples. 

12 apples * $0.50 per apple =



CoT:
To find out how much change you get, we need to calculate the total cost of the apples and then subtract that from the $10 bill.

Step 1: Calculate the total cost of the apples.
- Each apple costs $0.50.
- You buy 12 apples.
- Total cost = Number of apples * Cost per apple
- Total cost = 12 * $0.50
- Total cost = $6.00

Step 2: Calculate the change.
- You pay with a $10 bill.
- Change = Amount paid - Total cost
- Change = $10 - $6.00
- Change = $4.00

So, you get $4.00 as change.


## Zero-Shot CoT: 'Let's think step by step'

In [4]:
logical = "If all Bloops are Razzles, and all Razzles are Lazzles, are all Bloops definitely Lazzles?"

direct  = chat([{"role":"user","content":logical}], temperature=0, max_tokens=30)
zs_cot  = chat([{"role":"user","content":logical + "\n\nLet\'s think step by step."}],
               temperature=0, max_tokens=150)

print("Direct:", direct.strip())
print()
print("Zero-Shot CoT:")
print(zs_cot.strip())


Direct: To determine if all Bloops are Lazzles, we need to analyze the given information:

1. All Bloops are Razzles.
2

Zero-Shot CoT:
To determine if all Bloops are Lazzles, let's break it down step by step:

1. All Bloops are Razzles. This means that every Bloop is also a Razzle.
2. All Razzles are Lazzles. This means that every Razzle is also a Lazzle.

Now, let's combine these two statements:

- If all Bloops are Razzles, and all Razzles are Lazzles, then it logically follows that all Bloops are also Lazzles.

So, yes, all Bloops are indeed Lazzles.


## Few-Shot CoT: Examples with Reasoning

In [5]:
fewshot_cot = (
    "Solve the problem. Show your reasoning steps.\n\n"
    "Problem: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 balls. How many balls does he have?\n"
    "Reasoning: Roger starts with 5 balls. 2 cans * 3 balls/can = 6 new balls. 5 + 6 = 11 total.\n"
    "Answer: 11\n\n"
    "Problem: The cafeteria had 23 apples. If they used 20 for lunch and bought 6 more, how many do they have?\n"
    "Reasoning: Start with 23. Used 20: 23 - 20 = 3. Bought 6: 3 + 6 = 9.\n"
    "Answer: 9\n\n"
    "Problem: A farmer has 17 sheep. All but 9 die. How many sheep are left?\n"
    "Reasoning:"
)
result = chat([{"role":"user","content":fewshot_cot}], temperature=0, max_tokens=100)
print(result.strip())


To solve the problem, we need to subtract the number of sheep that died from the total number of sheep the farmer had.

Reasoning: 
1. Start with 17 sheep.
2. All but 9 died, which means 17 - 9 = 8 sheep are left.
3. Therefore, the farmer has 8 sheep left.

Answer: 8


## CoT for Multi-Step Reasoning

In [6]:
problem = (
    "A train leaves City A at 9:00 AM going 60 mph.\n"
    "Another train leaves City B at 10:00 AM going 80 mph toward City A.\n"
    "The cities are 280 miles apart.\n"
    "At what time do they meet?\n\n"
    "Think through this step by step."
)
result = chat([{"role":"user","content":problem}], temperature=0, max_tokens=300)
print(result.strip())


To find the time when the two trains meet, we need to calculate the distance covered by each train before they meet and then find the time it takes for them to cover that distance.

1. The first train leaves City A at 9:00 AM and travels at 60 mph. By the time the second train leaves City B at 10:00 AM, the first train has already traveled for 1 hour.

Distance covered by the first train in 1 hour = Speed * Time = 60 mph * 1 hour = 60 miles

Remaining distance between the two trains = Total distance - Distance covered by the first train
= 280 miles - 60 miles
= 220 miles

2. Now, both trains are moving towards each other. The relative speed of the two trains is the sum of their individual speeds.

Relative speed = Speed of first train + Speed of second train
= 60 mph + 80 mph
= 140 mph

3. Time taken for the two trains to cover the remaining distance = Distance / Relative speed
= 220 miles / 140 mph
= 1.57 hours (approximately)

4. Since the second train left City B at 10:00 AM, we nee

## CoT for Commonsense Reasoning

In [7]:
commonsense = (
    "Question: Would a pear sink or float in water?\n"
    "Let\'s think step by step.\n\n"
    "Step 1: Consider the density of a pear.\n"
    "Step 2: Compare to the density of water (1 g/cm3).\n"
    "Step 3: If object density < water density, it floats.\n"
    "Answer:"
)
result = chat([{"role":"user","content":commonsense}], temperature=0, max_tokens=100)
print(result.strip())


To determine whether a pear would sink or float in water, let's follow the steps:

Step 1: Consider the density of a pear.
Pears are generally made up of about 84% water and 16% solids (cell walls, fibers, etc.). The density of a pear is approximately 0.97-0.99 g/cm3, which is close to the density of water (1 g/cm3).

Step 2: Compare to the density of water (1 g
